# 25 Method Evaluation

Purpose: compare pre-fusion methods directly, without letting context-only rows dominate the result. This is where the 480 / 1086 style video-derived subsets are surfaced.

Runtime: CPU.

Outputs:
- `reports/method_evaluation/{report_id}/index.html`
- `reports/method_evaluation/{report_id}/summary.json`
- `reports/method_evaluation/{report_id}/tables/method_metrics.csv`
- `reports/method_evaluation/{report_id}/tables/same_sample_intersection_metrics.csv`
- `reports/method_evaluation/{report_id}/tables/sample_counts.csv`

Run this after 13/18/19/23/24 as methods become available. Fusion is not required.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id', 'method_evaluation_mlb_2024_2026_v2')
print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('REPORT_ID =', REPORT_ID)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)


In [ ]:
from sport_pipeline.reports.method_evaluation import build_method_evaluation_report

# Include context and fusion so the same report compares each method and the fused result. Turn these off only for visual-only appendices.
INCLUDE_CONTEXT_REFERENCE = True
INCLUDE_FUSION_REFERENCE = True

outputs = build_method_evaluation_report(
    BASE_DIR,
    RUN_PROFILE,
    report_id=REPORT_ID,
    include_context_reference=INCLUDE_CONTEXT_REFERENCE,
    include_fusion_reference=INCLUDE_FUSION_REFERENCE,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
summary = json.loads(outputs['summary'].read_text(encoding='utf-8'))
print('sample_counts preview')
for row in summary.get('sample_counts', [])[:30]:
    print(row)
print('method_metrics preview')
for row in summary.get('method_metrics', [])[:30]:
    print(row)
print(f'NEXT: use reports/method_evaluation/{REPORT_ID}/index.html for pre-fusion method results, then 15/22 if you want final fusion and paper-style bundle.')
